In [1]:
# download minsearch.py
# if you download yet, you can comment this cell

# !wget https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py

In [12]:
import minsearch
import json
import os

In [4]:
with open('../data/documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [5]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['category'] = course_dict['category']
        documents.append(doc)

In [6]:
documents[0]

{'section': 'Experiência Profissional',
 'question': 'Como destacar meu papel em projetos de ciência de dados?',
 'text': 'Descreva as ferramentas utilizadas, problemas resolvidos, resultados alcançados e impacto nos negócios.',
 'category': 'Curriculo'}

In [7]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["category"]
)

In [8]:
question = 'Como posso melhorar meu LinkeDIN?'

In [9]:
index.fit(documents)

In [10]:
from openai import OpenAI

In [13]:
client = OpenAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    base_url="https://api.mistral.ai/v1" 
)

In [14]:
response = client.chat.completions.create(
    model='mistral-medium',
    messages=[{"role": "user", "content": question}]
)

response.choices[0].message.content

'Existem várias maneiras de melhorar seu perfil no LinkedIn:\n\n1. Complete todo o seu perfil: Certifique-se de preencher todas as seções do seu perfil, incluindo experiência profissional, educação, habilidades e recomendações. Isso ajudará os recrutadores a entender melhor suas qualificações e experiências.\n2. Adicione uma foto profissional: Uma foto profissional ajudará a dar uma boa impressão aos recrutadores e conexões.\n3. Personalize seu resumo: Escreva um resumo claro e conciso que destache suas principais realizações e habilidades.\n4. Adicione habilidades relevantes: Adicione habilidades relevantes para o tipo de trabalho que você está procurando. Isso ajudará os recrutadores a encontrá-lo mais facilmente.\n5. Solicite recomendações: Solicite recomendações de ex-empregadores, colegas ou professores. Isso ajudará a construir sua reputação profissional.\n6. Participe de grupos: Participe de grupos relevantes para sua área de atuação e participe ativamente das discussões. Isso p

In [ ]:
def search(query):
    boost = {'questions': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'category': 'Curriculo'},
        boost_dict=boost,
        num_results=10
    )

    return results

In [ ]:
def build_prompt(query, search_results):
    prompt_template = """
        Você será um Chat Assistente. Responsa a QUESTION baseado no CONTEXT da base de dados FAQ.
        Use apenas os fatos do CONTEXT quando responder a QUESTION.

        QUESTION: {question}

        CONTEXT: {context}
""".strip()
    
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [ ]:
def llm(prompt):
    response = client.chat.completions.create(
        model='mistral-medium',
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
query = 'O que posso colocar ou usar no meu GitHub?'

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [ ]:
rag(query)

'Você pode usar e incluir links no seu perfil do GitHub, como para sua página no LinkedIn, portafolio online e projetos relevantes que contenham exemplos de seu trabalho. No entanto, evite colocar informações pessoais desnecessárias, como CPF ou RG, para proteger sua privacidade. Além disso, não é recomendado incluir todos os cursos que já fez, apenas aqueles recentes e relevantes para a vaga. Em vez de um objetivo profissional, considere usar um resumo profissional bem escrito focando em suas qualificações e o que você pode oferecer.\n\nEvite colocar pretensão salarial no GitHub ou em qualquer outro tipo de perfil profissional, a menos que seja solicitado especificamente pela vaga. Nesse caso, deixe essa discussão para a entrevista ou uma etapa posterior do processo de seleção. Você também não precisa incluir referências no GitHub, mas pode mencionar contribuições relevantes para projetos de código aberto ou outras realizações profissionais que demonstram suas habilidades e experiênci

In [ ]:
rag('Posso colocar meus projetos pessoais no GitHub?')

'Resposta: Sim, você pode e deve colocar seus projetos pessoais no GitHub. Isso ajudará a demonstrar suas habilidades práticas e conhecimentos técnicos. Certifique-se de incluir links para os projetos no seu currículo, destacando a stack usada, os objetivos do projeto, a estrutura de código e qualquer documentação ou testes implementados.'

In [ ]:
from elasticsearch import Elasticsearch

In [ ]:
es_client = Elasticsearch('http://localhost:9200')

In [ ]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "category": {"type": "keyword"} 
        }
    }
}

index_name = "category-question"

es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'category-question'})

In [ ]:
documents[0]

{'section': 'Experiência Profissional',
 'question': 'Como destacar meu papel em projetos de ciência de dados?',
 'text': 'Descreva as ferramentas utilizadas, problemas resolvidos, resultados alcançados e impacto nos negócios.',
 'category': 'Curriculo'}

In [ ]:
from tqdm.auto import tqdm

In [ ]:
for foc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

100%|██████████| 313/313 [00:04<00:00, 64.33it/s]


In [ ]:
query = 'Mesmo sem experiência profissional, posso colocar alguma experiência no meu Curriculo de algum projeto?'

In [ ]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^7", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "category": "Curriculo"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    
    result_docs = []
    
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

In [ ]:
def rag(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [ ]:
print(rag(query))

FAQ:
1. Posso incluir experiências não remuneradas em meu currículo?
Resposta: Sim, você pode incluir experiências não remuneradas em seu currículo, como estágios, voluntariados ou projetos pessoais. Isso pode ajudar a demonstrar suas habilidades e experiências práticas, mesmo sem ter experiência profissional remunerada.
2. Como listar projetos pessoais em meu currículo?
Resposta: Para listar projetos pessoais em seu currículo, você pode criar uma seção separada chamada "Projetos" ou incluí-los na seção de experiência, especificando que se trata de um projeto pessoal. Inclua detalhes sobre o projeto, como o objetivo, as habilidades utilizadas e os resultados alcançados.
3. Como listar voluntariados em meu currículo?
Resposta: Para listar voluntariados em seu currículo, crie uma seção separada chamada "Voluntariado" ou inclua-os na seção de experiência, especificando que se trata de um trabalho voluntário. Inclua detalhes sobre o voluntariado, como a organização, a duração, as responsab